# CSE — Portfolio Optimization Baseline

Classical portfolio construction on CSE equities.

**Covers:**
1. Mean-Variance (Markowitz) optimization — max Sharpe
2. Minimum Variance portfolio
3. Equal-Weight benchmark
4. Out-of-sample performance comparison

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from scipy.optimize import minimize

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

RISK_FREE = 0.08 / 252  # ~8% annual CBSL policy rate, daily

In [ ]:
df = pd.read_parquet('../data/published/cse_unified.parquet')
df['date'] = pd.to_datetime(df['date'])

# Select top N stocks by number of trading days (most complete history)
TOP_N = 30
eligible = (
    df[df['is_trading_day']]
    .groupby('symbol')['date']
    .count()
    .sort_values(ascending=False)
    .head(TOP_N)
    .index.tolist()
)
print(f'Selected {len(eligible)} symbols for portfolio construction')

# Pivot to wide returns matrix (only trading days)
prices = (
    df[df['symbol'].isin(eligible)]
    .pivot_table(index='date', columns='symbol', values='adj_close')
    .sort_index()
)
# Keep only dates where at least 80% of stocks have data
prices = prices.dropna(thresh=int(0.8 * len(eligible)))
returns = prices.pct_change().dropna(how='all')

print(f'Returns matrix: {returns.shape}')
print(f'Date range: {returns.index.min().date()} to {returns.index.max().date()}')

## 1. Train / Test Split (80 / 20 chronological)

In [ ]:
split_idx = int(len(returns) * 0.80)
train_ret = returns.iloc[:split_idx]
test_ret  = returns.iloc[split_idx:]

# Drop columns with too many NaNs in training set
valid_cols = train_ret.columns[train_ret.isnull().mean() < 0.20]
train_ret  = train_ret[valid_cols].fillna(0)
test_ret   = test_ret[valid_cols].fillna(0)

mu    = train_ret.mean().values        # expected daily returns
sigma = train_ret.cov().values         # covariance matrix
n     = len(valid_cols)

print(f'Train: {len(train_ret)} days | Test: {len(test_ret)} days | Stocks: {n}')

## 2. Optimization Helpers

In [ ]:
def portfolio_stats(w, mu, sigma):
    ret = w @ mu
    vol = np.sqrt(w @ sigma @ w)
    sharpe = (ret - RISK_FREE) / (vol + 1e-12)
    return ret, vol, sharpe

constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
bounds = [(0, 0.20)] * n  # max 20% per stock (long-only)
w0 = np.ones(n) / n

def max_sharpe_weights(mu, sigma):
    def neg_sharpe(w):
        ret, vol, _ = portfolio_stats(w, mu, sigma)
        return -(ret - RISK_FREE) / (vol + 1e-12)
    res = minimize(neg_sharpe, w0, method='SLSQP',
                   bounds=bounds, constraints=constraints,
                   options={'maxiter': 1000, 'ftol': 1e-9})
    return res.x

def min_variance_weights(sigma):
    def port_vol(w):
        return np.sqrt(w @ sigma @ w)
    res = minimize(port_vol, w0, method='SLSQP',
                   bounds=bounds, constraints=constraints,
                   options={'maxiter': 1000, 'ftol': 1e-9})
    return res.x

print('Solving Max-Sharpe portfolio...')
w_sharpe = max_sharpe_weights(mu, sigma)
print('Solving Min-Variance portfolio...')
w_minvar = min_variance_weights(sigma)
w_eq = np.ones(n) / n

for name, w in [('Equal-Weight', w_eq), ('Max-Sharpe', w_sharpe), ('Min-Variance', w_minvar)]:
    r, v, s = portfolio_stats(w, mu, sigma)
    print(f'{name:15s}  ann_ret={r*252:.3f}  ann_vol={v*np.sqrt(252):.3f}  sharpe={s*np.sqrt(252):.3f}')

## 3. Efficient Frontier

In [ ]:
target_returns = np.linspace(mu.min(), mu.max(), 60)
frontier_vols  = []

for target_r in target_returns:
    cons = [
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        {'type': 'eq', 'fun': lambda w, t=target_r: w @ mu - t},
    ]
    res = minimize(lambda w: np.sqrt(w @ sigma @ w), w0,
                   method='SLSQP', bounds=bounds, constraints=cons,
                   options={'maxiter': 500, 'ftol': 1e-9})
    frontier_vols.append(np.sqrt(res.x @ sigma @ res.x) if res.success else np.nan)

frontier_vols = np.array(frontier_vols)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(frontier_vols * np.sqrt(252), target_returns * 252,
        lw=2, color='steelblue', label='Efficient Frontier')

for name, w, color in [('Equal-Weight', w_eq, 'grey'),
                         ('Max-Sharpe',   w_sharpe, 'green'),
                         ('Min-Variance', w_minvar, 'red')]:
    r, v, _ = portfolio_stats(w, mu, sigma)
    ax.scatter(v*np.sqrt(252), r*252, s=80, color=color, zorder=5, label=name)

ax.set_xlabel('Annualised Volatility')
ax.set_ylabel('Annualised Return')
ax.set_title('CSE Efficient Frontier (Top 30 stocks, training period)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Out-of-Sample Cumulative Return

In [ ]:
portfolios = {
    'Equal-Weight':  w_eq,
    'Max-Sharpe':    w_sharpe,
    'Min-Variance':  w_minvar,
}

fig, ax = plt.subplots(figsize=(14, 5))

for name, w in portfolios.items():
    port_ret = (test_ret.values @ w)
    cum = (1 + port_ret).cumprod()
    ax.plot(test_ret.index, cum, lw=1.5, label=name)

ax.set_ylabel('Cumulative return')
ax.set_title('Out-of-Sample Portfolio Performance')
ax.legend()
plt.tight_layout()
plt.show()

print('\nOut-of-sample performance:')
for name, w in portfolios.items():
    port_ret = test_ret.values @ w
    ann_ret = port_ret.mean() * 252
    ann_vol = port_ret.std() * np.sqrt(252)
    sharpe  = (ann_ret - RISK_FREE * 252) / (ann_vol + 1e-12)
    cum     = (1 + port_ret).prod() - 1
    print(f'{name:15s}  ann_ret={ann_ret:.3f}  ann_vol={ann_vol:.3f}  sharpe={sharpe:.2f}  total_return={cum:.3f}')